<a href="https://colab.research.google.com/github/sfs-code/ai-image-dataset/blob/main/comunity-forensics-eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/JeongsooP/Community-Forensics

Cloning into 'Community-Forensics'...
remote: Enumerating objects: 73, done.
remote: Counting objects: 100% (73/73), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 73 (delta 30), reused 50 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (73/73), 6.90 MiB | 20.89 MiB/s, done.
Resolving deltas: 100% (30/30), done.


In [2]:
!git clone https://github.com/sfs-code/ai-image-dataset

Cloning into 'ai-image-dataset'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 59 (delta 0), reused 0 (delta 0), pack-reused 54 (from 2)
Receiving objects: 100% (59/59), 60.68 MiB | 50.23 MiB/s, done.


In [4]:
import sys
import os
subdirectory_path = '/content/Community-Forensics'
# Insert the path at a specific index (e.g., index 0 for highest priority)
sys.path.insert(0, subdirectory_path)

# Verify that the path has been added
print(sys.path)

!pip install torchmetrics

from google.colab import userdata
userdata.get('HF_TOKEN')

import torch
print(torch.cuda.is_available())


['/content/Community-Forensics', '/content/Community-Forensics', '/content', '/env/python', '/usr/lib/python312.zip', '/usr/lib/python3.12', '/usr/lib/python3.12/lib-dynload', '', '/usr/local/lib/python3.12/dist-packages', '/usr/lib/python3/dist-packages', '/usr/local/lib/python3.12/dist-packages/IPython/extensions', '/root/.ipython']
True


In [5]:
import models
import torch
import PIL.Image as Image
import dataprocessor_hf as dphf

In [6]:
data_processor = dphf.CommForImageProcessor.from_pretrained('OwensLab/commfor-data-preprocessor', size=384)
model_384 = models.ViTClassifier.from_pretrained('OwensLab/commfor-model-384').to('cuda')

preprocessor_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/88.8M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/87.3M [00:00<?, ?B/s]

In [16]:
#directory_path = '/content/Community-Forensics/test_images' # Replace with your directory path
directory_path = '/content/ai-image-dataset/images'

test_images = []

def collect_images_recursively(current_path, image_list):
    with os.scandir(current_path) as entries:
        for entry in entries:
            if entry.is_file() and entry.name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
                print(entry.path) # entry.path gives the full path
                image_list.append(Image.open(entry.path))
            elif entry.is_dir():
                collect_images_recursively(entry.path, image_list)

collect_images_recursively(directory_path, test_images)

# You can uncomment the line below if you want to print something about the collected images,
# but Image.Image objects do not have a .text attribute.
# For example, to print the filenames:
# [print(img.filename) for img in test_images]


/content/ai-image-dataset/images/img-003/B-img-003-d-nm.png
/content/ai-image-dataset/images/img-003/C-img-003-d-nm-cp.png
/content/ai-image-dataset/images/img-003/D-img-003-d-nm-cp-br.png
/content/ai-image-dataset/images/img-003/A-img-003-x-nm.png
/content/ai-image-dataset/images/img-008/D-img-008-d-nm-cp-br.png
/content/ai-image-dataset/images/img-008/A-img-008-x-nm.png
/content/ai-image-dataset/images/img-008/C-img-008-d-nm-cp.png
/content/ai-image-dataset/images/img-008/B-img-008-d-nm.png
/content/ai-image-dataset/images/img-005/A-img-005-x-nm.png
/content/ai-image-dataset/images/img-005/D-img-005-d-nm-cp-br.png
/content/ai-image-dataset/images/img-005/C-img-005-d-nm-cp.png
/content/ai-image-dataset/images/img-005/B-img-005-d-nm.png
/content/ai-image-dataset/images/img-006/A-img-006-x-nm.png
/content/ai-image-dataset/images/img-006/B-img-006-d-nm.png
/content/ai-image-dataset/images/img-006/D-img-006-d-nm-cp-br.png
/content/ai-image-dataset/images/img-006/C-img-006-d-nm-cp.png
/con

In [19]:
import torch
import pandas as pd
import os

torch.set_printoptions(sci_mode=False, precision=5)
input = data_processor(test_images, mode='test')
out = model_384(input['pixel_values'].to('cuda'))
results = torch.sigmoid(out)

print("Results for 384-input size model:")

# Extract filenames from test_images for better readability
image_filenames = [os.path.basename(img.filename) for img in test_images]

# Create a list of dictionaries for DataFrame
data_for_df = []
for i, result in enumerate(results):
    filename = image_filenames[i]
    score = result.item()
    print(f"Image: {filename}, Result: {score:.5f}")
    data_for_df.append({'Image Filename': filename, 'Result': f"{score:.5f}"})

# Create pandas DataFrame
results_df = pd.DataFrame(data_for_df)
ts = datetime.now().timestamp()

# Export to CSV using pandas
csv_file_path = f'image_results_{ts}.csv'
results_df.to_csv(csv_file_path, index=False)

print(f"\nResults successfully written to {csv_file_path} using pandas.")

OutOfMemoryError: CUDA out of memory. Tried to allocate 110.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 5.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.30 GiB is allocated by PyTorch, and 133.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://pytorch.org/docs/stable/notes/cuda.html#environment-variables)